In [1]:
pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 1.4 MB/s  0:00:06m0:00:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 2.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 3.6 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17/17 [streamlit]17 [streamlit]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import ast
from sklearn.preprocessing import LabelEncoder
import urllib.request
import tarfile
import os
url          = "https://sid.erda.dk/share_redirect/c7LF5NaYvH"
archive_name = "data.tar.xz"
path_elec    = "data/QMdata4ML/df_elec.csv.gz"

if not os.path.exists(path_elec):
    if not os.path.exists(archive_name):
        print("Downloading archive (~1.1 GB)...")
        urllib.request.urlretrieve(url, archive_name)
        print("Download complete.")
    print("Extracting archive...")
    with tarfile.open(archive_name, "r:xz") as tar:
        tar.extractall()
    print("Extraction complete.")

df = pd.read_csv(path_elec)
print(f"Loaded {len(df)} rows.")
try:
    display(df.head())
except NameError:
    print(df.head())


# 1. TEACHER MODEL TRAINING

print("🎓 STEP 1: Training the Master Model (XGBoost)...")

# Site encoding
encoder = LabelEncoder()
df['Site_Encoded'] = encoder.fit_transform(df['elec_sites'])

# Parsing charges (in case they are string-formatted)
def parse_charges(charge_data):
    if isinstance(charge_data, str): return ast.literal_eval(charge_data)
    return charge_data

df['charges_list'] = df['elec_GCS_3_cm5'].apply(parse_charges)
charges_df = pd.DataFrame(df['charges_list'].tolist(), index=df.index).add_prefix('charge_')

# X/y preparation for the Teacher
X_master = pd.concat([df[['Site_Encoded']], charges_df], axis=1)
y_master = df['MAA_values']

# Optimized parameters for the Teacher
xgb_master = xgb.XGBRegressor(
    n_estimators=1800, max_depth=11, learning_rate=0.0617, 
    tree_method='hist', n_jobs=7, random_state=42
)
xgb_master.fit(X_master, y_master)

# Distillation: Creating targets for the student
df['Teacher_MAA_Predictions'] = xgb_master.predict(X_master)
print("✅ Knowledge Distillation targets successfully generated.")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import ast
import optuna
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

# Load dataset
df = pd.read_parquet('df_elec.parquet')

# 1. TEACHER MODEL TRAINING
print("🎓 STEP 1: Training the Master Model (XGBoost)...")

encoder = LabelEncoder()
df['Site_Encoded'] = encoder.fit_transform(df['elec_sites'])

def parse_charges(charge_data):
    if isinstance(charge_data, str): 
        return ast.literal_eval(charge_data)
    return charge_data

df['charges_list'] = df['elec_GCS_3_cm5'].apply(parse_charges)
charges_df = pd.DataFrame(df['charges_list'].tolist(), index=df.index).add_prefix('charge_')

X_master = pd.concat([df[['Site_Encoded']], charges_df], axis=1)
y_master = df['MAA_values']

xgb_master = xgb.XGBRegressor(
    n_estimators=1800, max_depth=11, learning_rate=0.0617, 
    tree_method='hist', n_jobs=7, random_state=42
)
xgb_master.fit(X_master, y_master)

df['Teacher_MAA_Predictions'] = xgb_master.predict(X_master)
print("✅ Knowledge Distillation targets successfully generated.")

# 2. HYPERPARAMETER OPTIMIZATION (OPTUNA)
print("\n🔍 STEP 2: Launching Optuna Search (Limited to 15 Trials)...")

unique_smiles = df['smiles'].unique()
np.random.seed(42)
sampled_smiles = np.random.choice(unique_smiles, size=min(2500, len(unique_smiles)), replace=False)
df_study = df[df['smiles'].isin(sampled_smiles)].reset_index(drop=True)

print(f"🎯 Optuna Study Dataset: {df_study['smiles'].nunique()} unique molecules across {len(df_study)} sites.")

def objective(trial):
    rad_g = trial.suggest_int('rad_global', 1, 3)
    rad_l = trial.suggest_int('rad_local', 2, 6)
    n_bits = trial.suggest_categorical('n_bits', [2048, 4096])
    
    xgb_params = {
        'n_estimators': 800,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 6, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'tree_method': 'hist',
        'n_jobs': 7,
        'random_state': 42
    }

    def get_fp(smiles, idx):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if mol:
                g = AllChem.GetMorganFingerprintAsBitVect(mol, radius=rad_g, nBits=n_bits)
                l = AllChem.GetMorganFingerprintAsBitVect(mol, radius=rad_l, nBits=n_bits, fromAtoms=[int(idx)])
                return list(g) + list(l)
        except:
            pass
        return [0] * (n_bits * 2)

    X_study = np.array([get_fp(s, i) for s, i in zip(df_study['smiles'], df_study['elec_sites'])], dtype=np.int8)
    y_study = df_study['Teacher_MAA_Predictions'].values
    groups_study = df_study['smiles'].values 

    gkf = GroupKFold(n_splits=3)
    scores = []

    for train_idx, val_idx in gkf.split(X_study, y_study, groups=groups_study):
        X_train, X_val = X_study[train_idx], X_study[val_idx]
        y_train, y_val = y_study[train_idx], y_study[val_idx]

        model = xgb.XGBRegressor(**xgb_params)
        model.fit(X_train, y_train)
        
        preds = model.predict(X_val)
        scores.append(r2_score(y_val, preds))

    return np.mean(scores)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15)

print("\n🏆 OPTUNA BEST HYPERPARAMETERS FOUND")
print(study.best_params)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from joblib import Parallel, delayed
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib

# --- 1.PARAMETERS ---
BEST_RAD_G = 1
BEST_RAD_L = 2
BEST_BITS = 4096

BEST_XGB_PARAMS = {
    'n_estimators': 2000,
    'learning_rate': 0.074,
    'max_depth': 11,
    'min_child_weight': 2,
    'subsample': 0.78,
    'colsample_bytree': 0.952,
    'tree_method': 'hist',
    'device': 'cpu', 
    'n_jobs': 7,
    'random_state': 42
}

# --- 2. FEATURE GENERATION ---
def get_final_fp(smiles, site_idx):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            g = AllChem.GetMorganFingerprintAsBitVect(mol, radius=BEST_RAD_G, nBits=BEST_BITS)
            l = AllChem.GetMorganFingerprintAsBitVect(mol, radius=BEST_RAD_L, nBits=BEST_BITS, fromAtoms=[int(site_idx)])
            return list(g) + list(l)
    except:
        pass
    return [0] * (BEST_BITS * 2)

print(f"🚀 Generating fingerprints for the entire dataset ({len(df)} sites)...")
results = Parallel(n_jobs=7, prefer="threads", verbose=10)(
    delayed(get_final_fp)(s, idx) for s, idx in zip(df['smiles'], df['elec_sites'])
)

X = np.array(results, dtype=np.int8)
y_teacher = df['Teacher_MAA_Predictions'].values.astype('float32')
y_labo = df['MAA_values'].values.astype('float32')

# --- 3. CANONICAL SPLIT BY MOLECULE (80 / 10 / 10) ---
print("\n📦 Splitting data by molecule (GroupSplit)...")

# Step 1: Isolate 10% for the final Test set
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
train_val_idx, test_idx = next(gss_test.split(X, y_teacher, groups=df['smiles']))

X_train_val, X_test = X[train_val_idx], X[test_idx]
y_tv_teacher, y_test_teacher = y_teacher[train_val_idx], y_teacher[test_idx]
y_tv_labo, y_test_labo = y_labo[train_val_idx], y_labo[test_idx]
groups_tv = df['smiles'].iloc[train_val_idx]

# Step 2: In the remaining 90%, isolate 11.11% for Val (which equals 10% of the global total)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1111, random_state=42)
train_idx, val_idx = next(gss_val.split(X_train_val, y_tv_teacher, groups=groups_tv))

X_train, X_val = X_train_val[train_idx], X_train_val[val_idx]
y_train, y_val = y_tv_teacher[train_idx], y_tv_teacher[val_idx]

# Security check for unique molecules
train_smiles = set(df['smiles'].iloc[train_val_idx].iloc[train_idx])
val_smiles = set(df['smiles'].iloc[train_val_idx].iloc[val_idx])
test_smiles = set(df['smiles'].iloc[test_idx])

print(f"  ↳ Train : {len(X_train)} sites ({len(train_smiles)} molecules)")
print(f"  ↳ Val   : {len(X_val)} sites ({len(val_smiles)} molecules)")
print(f"  ↳ Test  : {len(X_test)} sites ({len(test_smiles)} molecules)")
assert train_smiles.isdisjoint(val_smiles) and train_smiles.isdisjoint(test_smiles), "❌ Leak detected between sets!"
print("  ✓ Canonical split validated without leakage.")

# --- 4. CHAMPION STUDENT TRAINING ---
print("\n🔥 Training Champion Student in progress...")
final_eleve = xgb.XGBRegressor(**BEST_XGB_PARAMS)

# Direct use of the 10% validation set for tracking/implicit early stopping
final_eleve.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

# --- 5. FINAL VERDICT ---
y_pred = final_eleve.predict(X_test)

r2_vs_teacher = r2_score(y_test_teacher, y_pred)
r2_vs_labo = r2_score(y_test_labo, y_pred)

print("\n" + "="*60)
print("🏆 FINAL OPTIMIZED MODEL RESULTS (CANONICAL SPLIT)")
print("="*60)
print(f"📈 R² (Student vs Teacher) on Test : {r2_vs_teacher:.4f}")
print(f"🎯 R² (Student vs Lab)     on Test : {r2_vs_labo:.4f}")
print(f"📉 RMSE (vs Lab)           on Test : {np.sqrt(mean_squared_error(y_test_labo, y_pred)):.2f}")
print(f"📉 MAE (vs Lab)            on Test : {mean_absolute_error(y_test_labo, y_pred):.2f}")
print("="*60)

# Save the model
joblib.dump(final_eleve, 'model_eleve_champion.pkl')
print("💾 Model saved as 'model_eleve_champion.pkl'")

🚀 Generating fingerprints for the entire dataset (534119 sites)...


[Parallel(n_jobs=7)]: Using backend ThreadingBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done   4 tasks      | elapsed:    0.0s
[Parallel(n_jobs=7)]: Done  11 tasks      | elapsed:    0.1s
[Parallel(n_jobs=7)]: Done  18 tasks      | elapsed:    0.1s
[Parallel(n_jobs=7)]: Done  27 tasks      | elapsed:    0.1s
[Parallel(n_jobs=7)]: Done  36 tasks      | elapsed:    0.1s
[Parallel(n_jobs=7)]: Done  47 tasks      | elapsed:    0.1s
[Parallel(n_jobs=7)]: Done  58 tasks      | elapsed:    0.2s
[Parallel(n_jobs=7)]: Done  71 tasks      | elapsed:    0.2s
[Parallel(n_jobs=7)]: Done  84 tasks      | elapsed:    0.3s
[Parallel(n_jobs=7)]: Done  99 tasks      | elapsed:    0.3s
[Parallel(n_jobs=7)]: Done 114 tasks      | elapsed:    0.4s
[Parallel(n_jobs=7)]: Done 131 tasks      | elapsed:    0.4s
[Parallel(n_jobs=7)]: Done 148 tasks      | elapsed:    0.5s
[Parallel(n_jobs=7)]: Done 167 tasks      | elapsed:    0.5s
[Parallel(n_jobs=7)]: Done 186 tasks      | elapsed:    0.6s
[Para


📦 Splitting data by molecule (GroupSplit)...
  ↳ Train : 427529 sites (37952 molecules)
  ↳ Val   : 53119 sites (4744 molecules)
  ↳ Test  : 53471 sites (4744 molecules)
  ✓ Canonical split validated without leakage.

🔥 Training Champion Student in progress...
[0]	validation_0-rmse:67.89826
[100]	validation_0-rmse:38.81112
[200]	validation_0-rmse:35.76807
[300]	validation_0-rmse:34.29458
[400]	validation_0-rmse:33.33578
[500]	validation_0-rmse:32.64197
[600]	validation_0-rmse:32.07754
[700]	validation_0-rmse:31.59741
[800]	validation_0-rmse:31.19645
[900]	validation_0-rmse:30.87100
[1000]	validation_0-rmse:30.58148
[1100]	validation_0-rmse:30.30645
[1200]	validation_0-rmse:30.06106
[1300]	validation_0-rmse:29.85308
[1400]	validation_0-rmse:29.65278
[1500]	validation_0-rmse:29.47118
[1600]	validation_0-rmse:29.30929
[1700]	validation_0-rmse:29.15434
[1800]	validation_0-rmse:29.00610
[1900]	validation_0-rmse:28.87996
[1999]	validation_0-rmse:28.75389

🏆 FINAL OPTIMIZED MODEL RESULTS (CA